**Instructions**:

* You may customize the filenames for the regression and classification results by setting your preferred values in the regr_filename and clf_filename variables in the next cell.

* If you are running the experiments script for the main results, ensure that:

    - The string "_main" is appended before the .csv extension in the respective filenames (e.g., "results_regression_main.csv") like the commented filenames of the next cell.

    - You only run the Main Results Sections of this notebook.

* Otherwise, all other sections should function normally without modifications.

In [ ]:
# regr_filename = "results_regression_main.csv"
# clf_filename = "results_classification_main.csv"

regr_filename = "results_regression.csv"
clf_filename = "results_classification.csv"

In [ ]:
import pandas as pd 
import numpy as np

# Labels Maps

In [ ]:
clf_method_to_display_label = {
    "MaskedNAMClassifier": "NAM",
    "PyGAMClassifier": "Spline",
    "NoInteractionsEBMClassifier": "EBM",
    "GAMBoostingClassifier": "GAM Boosting",
    "CALMClassifier": "CALM",
    "EBM2Classifier": "EBM2",
    "NodeGAM2Classifier": "NodeGAM2",
    "GAMINetClassifier": "GAMINet",
    "RFClassifier": "RF",
    "XGBClassifier": "XGB",
    "Black Box Model": "DNN",
    "DNNClassifier": "DNN",
}

regr_method_to_display_label = {
    "MaskedNAMRegressor": "NAM",
    "PyGAMRegressor": "Spline",
    "NoInteractionsEBMRegressor": "EBM",
    "GAMBoostingRegressor": "GAM Boosting",
    "CALMRegressor": "CALM",
    "EBM2Regressor": "EBM2",
    "NodeGAM2Regressor": "NodeGAM2",
    "GAMINetRegressor": "GAMINet",
    "RFRegressor": "RF",
    "XGBRegressor": "XGB",
    "Black Box Model": "DNN",
    "DNNRegressor": "DNN",
}

dataset_to_display_label = {
    "ConditionalInteractionRegression": "Synth (Conditional Interaction)",
    "ConditionalInteractionClassification": "Synth (Conditional Interaction)",
    "DoubleConditionalInteractionClassification": "Synth (Double Conditional Interaction)",
    "DoubleConditionalInteractionRegression": "Synth (Double Conditional Interaction)",
    "BikeSharing": "Bike Sharing",
    "CaliforniaHousing": "California Housing",
    "ParkinsonsMotor": "Parkinsons Motor",
    "ParkinsonsTotal": "Parkinsons Total",
    "SeoulBike": "Seoul Bike",
    "SkillCraft": "Skill Craft",
    "PMLB_APPENDICITIS": "Appendicitis",
    "PMLB_PHONEME": "Phoneme",
    "PMLB_SPECTF": "SPECTF",
    "PMLB_CHURN": "Churn",
}


ordered_methods = [
    "XGB",
    "NAM",
    "EBM",
    "CALM",
    "EBM2",
    "NodeGAM2",
    "GAMINet",
]

ordered_methods_without_blackbox = [
    "NAM",
    "EBM",
    "Spline",
    "CALM - NAM - PDP",
    "CALM - NAM - RHALE",
    "CALM - EBM - PDP",
    "CALM - EBM - RHALE",
    "CALM - Spline - PDP",
    "CALM - Spline - RHALE",
]

ordered_methods_without_blackbox_pdp = [
    "NAM",
    "EBM",
    "Spline",
    "CALM - NAM - PDP",
    "CALM - EBM - PDP",
    "CALM - Spline - PDP",
]

black_box_models = ["DNN", "XGB", "RF"]
gam = ["NAM", "EBM", "Spline"]
calm_methods = [
    "CALM",
    "CALM - DNN",
    "CALM - XGB",
    "CALM - RF",
    "CALM - NAM - PDP",
    "CALM - NAM - RHALE",
    "CALM - EBM - PDP",
    "CALM - EBM - RHALE",
    "CALM - Spline - PDP",
    "CALM - Spline - RHALE",
]
ga2m = ["EBM2", "NodeGAM2", "GAMINet"]

regr_datasets = [
    "BikeSharing",
    "CaliforniaHousing",
    "ParkinsonsMotor",
    "ParkinsonsTotal",
    "SeoulBike",
    "Wine",
    "Energy",
    "CCPP",
    "Electrical",
    "Elevators",
    "No2",
    "Sensory",
    "Airfoil",
    "SkillCraft",
    "Ailerons",
]

clf_datasets = [
    "Adult",
    "COMPAS",
    "HELOC",
    "MIMIC2",
    "PMLB_APPENDICITIS",
    "PMLB_PHONEME",
    "PMLB_SPECTF",
    "Magic",
    "Bank",
    "PMLB_CHURN",
]

region_detector_display_names = {
    "RegionalPDP": "PDP",
    "RegionalRHALE": "RHALE",
}

# Utils

In [ ]:
def better_score_symbol(calm, other_score, lower_is_better=False):
    if (not lower_is_better and calm > other_score) or (lower_is_better and calm < other_score):
        return "✓"  
    elif other_score - calm == 0.0:
        return "-"
    else:
        return "✗"  


def format_float(value, decimals=3, small_threshold=1e-3, max_decimals=10):
    if value == 0:
        return f"{0:.{decimals}f}"

    # Check if rounding would make it appear as 0
    if round(value, decimals) == 0:
        temp_decimals = decimals
        while round(value, temp_decimals) == 0 and temp_decimals < max_decimals:
            temp_decimals += 1
        return f"{value:.{temp_decimals}f}"

    return f"{value:.{decimals}f}"


def get_table_scores(
    df,
    score_label,
    fillna=True,
    calm_vs_all=False,
    ordered_methods=ordered_methods,
    decimals=3, 
    avg_per_method=False,
    lower_is_better=False,
):
    results = []
    maximize = score_label in ["accuracy", "f1", "balanced_accuracy", "r2"]
    
    for dataset in df["dataset"].unique():
        dataset_df = df[df["dataset"] == dataset]
        res = {"dataset": dataset}
            
        for method in ordered_methods:
            method_df = dataset_df[dataset_df["method"] == method]
            if method_df.empty:
                if fillna:
                    res[method] = "N/A"
                continue
            method_mean = method_df[f"{score_label}_mean"].values[0]
            method_std = method_df[f"{score_label}_std"].values[0]
            format_mean = format_float(method_mean, decimals=decimals)
            format_std = format_float(method_std, decimals=decimals, max_decimals=3)
            res[method] = f"{format_mean} ± {format_std}"
            res[f"{method}_mean"] = method_mean


        calm_score = dataset_df[dataset_df["method"].isin(calm_methods)][f"{score_label}_mean"].max() if maximize else \
            dataset_df[dataset_df["method"].isin(calm_methods)][f"{score_label}_mean"].min()
        
        calm_score = calm_score.round(decimals)
        
        if calm_vs_all:
            dataset_blackboxes = dataset_df[dataset_df["method"].isin(black_box_models)]

            if dataset_blackboxes.empty:
                res["CALM vs BlackBox"] = "N/A"
            else:
                blackbox_avg = dataset_blackboxes[f"{score_label}_mean"].round(decimals).mean().round(decimals)
                symbol = better_score_symbol(calm_score, blackbox_avg, lower_is_better=lower_is_better) 
                res["CALM vs BlackBox"] = symbol

            dataset_gams = dataset_df[dataset_df["method"].isin(gam)]
            if dataset_gams.empty:
                res["CALM vs GAM"] = "N/A"
            else:
                gam_avg = dataset_gams[f"{score_label}_mean"].round(decimals).mean().round(decimals)
                symbol = better_score_symbol(calm_score, gam_avg, lower_is_better=lower_is_better)
                res["CALM vs GAM"] = symbol

            dataset_ga2m = dataset_df[dataset_df["method"].isin(ga2m)]
            if dataset_ga2m.empty:
                res["CALM vs GA2M"] = "N/A"
            else:
                ga2m_avg = dataset_ga2m[f"{score_label}_mean"].round(decimals).mean().round(decimals)
                symbol = better_score_symbol(calm_score, ga2m_avg, lower_is_better=lower_is_better)
                res["CALM vs GA2M"] = symbol

        results.append(res)

    results_df = pd.DataFrame(results)
    if avg_per_method:
        avg_row = {}
        avg_row["dataset"] = "Average"
        for method in ordered_methods:
            if method not in results_df.columns:
                continue
            mean = results_df[f"{method}_mean"].astype(float).mean()
            avg_row[method] = format_float(mean, decimals=decimals)
        results_df = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

    results_df = results_df.drop(columns=[f"{method}_mean" for method in ordered_methods], errors="ignore")
    results_df["dataset"] = results_df["dataset"].str.replace("_", " ", regex=False)
    return results_df


def get_main_results(df):
    df_ = df[
        ((df['blackbox_model'] == "XGB") &
        (df['masked_gam'] == "EBM") &
        (df['region_detector'] == "RegionalPDP") &
        (df['pcg_drop_thres'] == 0.2)
        ) |
        (df['masked_gam'].isna() &
         df['method'].isin(ordered_methods)) 
    ]
    df_.loc[df_["method"] == "CALM - XGB", 'method'] = "CALM"
    return df_

# Main Results

## Regression

In [ ]:
df = pd.read_csv(regr_filename)

df['method']  = df['method'].replace(regr_method_to_display_label)
df['masked_gam'] = df['masked_gam'].replace(regr_method_to_display_label)
df['blackbox_model'] = df['blackbox_model'].replace(regr_method_to_display_label)
df['dataset'] = df['dataset'].replace(dataset_to_display_label)
df['method'] = df.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df['calm_params'] = df.apply(lambda x: f"{x['masked_gam']} - {x['region_detector']} - {x['pcg_drop_thres']}", axis=1)

# keep fixed combinations of calm_params
df_reg_main = get_main_results(df)
df_reg_main = df_reg_main[df_reg_main['method'].isin(ordered_methods)]

### RMSE

In [ ]:
rmse_df = get_table_scores(
    df_reg_main,
    "rmse",
    calm_vs_all=True,
    ordered_methods=ordered_methods,
    decimals=3,
    lower_is_better=True,
)
display(rmse_df)

### Interactions

In [ ]:
ga2m_calm_df = df_reg_main[df_reg_main["method"].isin(ga2m + calm_methods)]
interactions_df = get_table_scores(
    ga2m_calm_df,
    "num_interactions",
    fillna=False,
    ordered_methods=ordered_methods,
    calm_vs_all=False,
    decimals=1,
    avg_per_method=True,
)
display(interactions_df)

### Runtime

In [ ]:
time_df = get_table_scores(
    df_reg_main,
    "runtime_sec",
    fillna=False,
    ordered_methods=ordered_methods,
    decimals=0,
    avg_per_method=True,
)
display(time_df)

## Classification

In [ ]:
df = pd.read_csv(clf_filename)

df['method']  = df['method'].replace(clf_method_to_display_label)
df['masked_gam'] = df['masked_gam'].replace(clf_method_to_display_label)
df['blackbox_model'] = df['blackbox_model'].replace(clf_method_to_display_label)
df['dataset'] = df['dataset'].replace(dataset_to_display_label)
df['method'] = df.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df['calm_params'] = df.apply(lambda x: f"{x['masked_gam']} - {x['region_detector']} - {x['pcg_drop_thres']}", axis=1)

# keep fixed combinations of calm_params
df_reg_main = get_main_results(df)
df_reg_main = df_reg_main[df_reg_main['method'].isin(ordered_methods)]

### Accuracy

In [ ]:
acc_df = get_table_scores(
    df_reg_main,
    "accuracy",
    calm_vs_all=True,
    ordered_methods=ordered_methods,
    decimals=3,
)
display(acc_df)

### Interactions

In [ ]:
ga2m_calm_df = df_reg_main[df_reg_main["method"].isin(ga2m + calm_methods)]
interactions_df = get_table_scores(
    ga2m_calm_df,
    score_label="num_interactions",
    fillna=False,
    calm_vs_all=False,
    ordered_methods=ordered_methods,
    decimals=1,
    avg_per_method=True,
)
display(interactions_df)

### Runtime

In [ ]:
time_df = get_table_scores(
    df_reg_main,
    "runtime_sec",
    ordered_methods=ordered_methods,
    decimals=0,
    avg_per_method=True,
    fillna=False,
)
display(time_df)

# Detailed Results

## Regression

### DNN

In [ ]:
ordered_methods_dnn = ["DNN"] + ordered_methods_without_blackbox

df_dnn = pd.read_csv(regr_filename)
df_dnn['method']  = df_dnn['method'].replace(regr_method_to_display_label)
df_dnn['masked_gam'] = df_dnn['masked_gam'].replace(regr_method_to_display_label)
df_dnn['blackbox_model'] = df_dnn['blackbox_model'].replace(regr_method_to_display_label)
df_dnn['dataset'] = df_dnn['dataset'].replace(dataset_to_display_label)
df_dnn['method'] = df_dnn.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df_dnn['region_detector'] = df_dnn['region_detector'].replace(region_detector_display_names)
df_dnn = df_dnn[(df_dnn['blackbox_model'].isna()) | (df_dnn['blackbox_model'] == "DNN")].reset_index(drop=True)
df_dnn['method'] = df_dnn.apply(lambda x: f"CALM - {x['masked_gam']} - {x['region_detector']}" if x['method'] == "CALM - DNN" else x['method'], axis=1)
df_dnn = df_dnn[df_dnn['method'].isin(ordered_methods_dnn)]

rmse_df = get_table_scores(
    df_dnn,
    "rmse",
    ordered_methods=ordered_methods_dnn,
    decimals=3,
)
display(rmse_df)

### XGB

In [ ]:
ordered_methods_xgb = ["XGB"] + ordered_methods_without_blackbox_pdp

df_xgb = pd.read_csv(regr_filename)
df_xgb['method']  = df_xgb['method'].replace(regr_method_to_display_label)
df_xgb['masked_gam'] = df_xgb['masked_gam'].replace(regr_method_to_display_label)
df_xgb['blackbox_model'] = df_xgb['blackbox_model'].replace(regr_method_to_display_label)
df_xgb['dataset'] = df_xgb['dataset'].replace(dataset_to_display_label)
df_xgb['method'] = df_xgb.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df_xgb['region_detector'] = df_xgb['region_detector'].replace(region_detector_display_names)
df_xgb = df_xgb[(df_xgb['blackbox_model'].isna()) | (df_xgb['blackbox_model'] == "XGB")].reset_index(drop=True)
df_xgb['method'] = df_xgb.apply(lambda x: f"CALM - {x['masked_gam']} - {x['region_detector']}" if x['method'] == "CALM - XGB" else x['method'], axis=1)
df_xgb = df_xgb[df_xgb['method'].isin(ordered_methods_xgb)]

rmse_df = get_table_scores(
    df_xgb,
    "rmse",
    ordered_methods=ordered_methods_xgb,
    decimals=3,
)
display(rmse_df)

### RF

In [ ]:
ordered_methods_rf = ["RF"] + ordered_methods_without_blackbox_pdp

df_rf = pd.read_csv(regr_filename)
df_rf['method']  = df_rf['method'].replace(regr_method_to_display_label)
df_rf['masked_gam'] = df_rf['masked_gam'].replace(regr_method_to_display_label)
df_rf['blackbox_model'] = df_rf['blackbox_model'].replace(regr_method_to_display_label)
df_rf['dataset'] = df_rf['dataset'].replace(dataset_to_display_label)
df_rf['method'] = df_rf.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df_rf['region_detector'] = df_rf['region_detector'].replace(region_detector_display_names)
df_rf = df_rf[(df_rf['blackbox_model'].isna()) | (df_rf['blackbox_model'] == "RF")].reset_index(drop=True)
df_rf['method'] = df_rf.apply(lambda x: f"CALM - {x['masked_gam']} - {x['region_detector']}" if x['method'] == "CALM - RF" else x['method'], axis=1)
df_rf = df_rf[df_rf['method'].isin(ordered_methods_rf)]

rmse_df = get_table_scores(
    df_rf,
    "rmse",
    ordered_methods=ordered_methods_rf,
    decimals=3,
)
display(rmse_df)

## Classification

### DNN

In [ ]:
ordered_methods_dnn = ["DNN"] + ordered_methods_without_blackbox

df_dnn = pd.read_csv(clf_filename)
df_dnn['method']  = df_dnn['method'].replace(clf_method_to_display_label)
df_dnn['masked_gam'] = df_dnn['masked_gam'].replace(clf_method_to_display_label)
df_dnn['blackbox_model'] = df_dnn['blackbox_model'].replace(clf_method_to_display_label)
df_dnn['dataset'] = df_dnn['dataset'].replace(dataset_to_display_label)
df_dnn['method'] = df_dnn.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df_dnn['region_detector'] = df_dnn['region_detector'].replace(region_detector_display_names)
df_dnn = df_dnn[(df_dnn['blackbox_model'].isna()) | (df_dnn['blackbox_model'] == "DNN")].reset_index(drop=True)
df_dnn['method'] = df_dnn.apply(lambda x: f"CALM - {x['masked_gam']} - {x['region_detector']}" if x['method'] == "CALM - DNN" else x['method'], axis=1)
df_dnn = df_dnn[df_dnn['method'].isin(ordered_methods_dnn)]

acc_df = get_table_scores(
    df_dnn,
    "accuracy",
    ordered_methods=ordered_methods_dnn,
    decimals=3,
)
display(acc_df)

### XGB

In [ ]:
ordered_methods_xgb = ["XGB"] + ordered_methods_without_blackbox_pdp

df_xgb = pd.read_csv(clf_filename)
df_xgb['method']  = df_xgb['method'].replace(clf_method_to_display_label)
df_xgb['masked_gam'] = df_xgb['masked_gam'].replace(clf_method_to_display_label)
df_xgb['blackbox_model'] = df_xgb['blackbox_model'].replace(clf_method_to_display_label)
df_xgb['dataset'] = df_xgb['dataset'].replace(dataset_to_display_label)
df_xgb['method'] = df_xgb.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df_xgb['region_detector'] = df_xgb['region_detector'].replace(region_detector_display_names)
df_xgb = df_xgb[(df_xgb['blackbox_model'].isna()) | (df_xgb['blackbox_model'] == "XGB")].reset_index(drop=True)
df_xgb['method'] = df_xgb.apply(lambda x: f"CALM - {x['masked_gam']} - {x['region_detector']}" if x['method'] == "CALM - XGB" else x['method'], axis=1)
df_xgb = df_xgb[df_xgb['method'].isin(ordered_methods_xgb)]

acc_df = get_table_scores(
    df_xgb,
    "accuracy",
    ordered_methods=ordered_methods_xgb,
    decimals=3,
)
display(acc_df)

### RF

In [ ]:
ordered_methods_rf = ["RF"] + ordered_methods_without_blackbox_pdp

df_rf = pd.read_csv(clf_filename)
df_rf['method']  = df_rf['method'].replace(clf_method_to_display_label)
df_rf['masked_gam'] = df_rf['masked_gam'].replace(clf_method_to_display_label)
df_rf['blackbox_model'] = df_rf['blackbox_model'].replace(clf_method_to_display_label)
df_rf['dataset'] = df_rf['dataset'].replace(dataset_to_display_label)
df_rf['method'] = df_rf.apply(lambda x: f"{x['method']} - {x['blackbox_model']}" if x['method'] == "CALM" else x['method'], axis=1)
df_rf['region_detector'] = df_rf['region_detector'].replace(region_detector_display_names)
df_rf = df_rf[(df_rf['blackbox_model'].isna()) | (df_rf['blackbox_model'] == "RF")].reset_index(drop=True)
df_rf['method'] = df_rf.apply(lambda x: f"CALM - {x['masked_gam']} - {x['region_detector']}" if x['method'] == "CALM - RF" else x['method'], axis=1)
df_rf = df_rf[df_rf['method'].isin(ordered_methods_rf)]

acc_df = get_table_scores(
    df_rf,
    "accuracy",
    ordered_methods=ordered_methods_rf,
    decimals=3,
)
display(acc_df)